In [ ]:
import time
import numpy as np
import pickle
import pandas as pd
from scipy.stats import multivariate_normal, gaussian_kde
from sklearn.metrics import mutual_info_score, normalized_mutual_info_score
from sklearn.feature_selection import mutual_info_regression
from numba import jit
from numba import cuda

from scipy.integrate import quad

from scipy.stats import gaussian_kde

In [ ]:
with open("../transformed_event_logs/BPIC_2017_all_train.pickle", "rb") as f:
    event_log = pickle.load(f)
event_log

In [ ]:
list(event_log.columns)

In [ ]:
# numerical attributes : duration, seconds_in_day, day_in_week
numerical_attributes = [
    'duration_seconds',
    'seconds_in_day',
    'day_of_week',
    'case:RequestedAmount_start'
]


In [ ]:
transformed_event_log = event_log.copy()

for num_attr in numerical_attributes:
    transformed_event_log[num_attr] = np.log(transformed_event_log[num_attr]+1)
    transformed_event_log[num_attr] = (transformed_event_log[num_attr] - transformed_event_log[num_attr].mean()) / transformed_event_log[num_attr].std()

In [ ]:
transformed_event_log

## Mutual Information: Discrete - Discrete

In [ ]:
def MI_discrete_discrete(df, col, target_col):
    p_x = df[col].value_counts(normalize=True)
    p_y = df[target_col].value_counts(normalize=True)
    joint_counts = df.groupby([col, target_col]).size()
    p_xy = joint_counts / len(df)

    mi = 0.0
    for (x, y), pxy in p_xy.items():
        px = p_x[x]
        py = p_y[y]
        mi += pxy * np.log(pxy / (px * py))
    return mi


## Mutual Information: Discrete - Continuous

In [ ]:
def MI_discrete_continuous(df, col, target_col='duration_seconds',
                gridsize=4096,  # power of two → cheap FFT in KDE
                eps=1e-12):
    """
    Mutual information I(col ; target_col) for a *discrete* X=col and
    a *continuous* Y=target_col, using Gaussian KDE + log‑space integration.
    """
    # ------------------  Pre‑compute pieces that do not depend on x  ----------
    y_vals = df[target_col].to_numpy()
    y_min, y_max = y_vals.min(), y_vals.max()
    padding = 0.05 * (y_max - y_min)

    # Fixed grid avoids adaptive sampling where the KDE is unstable
    grid = np.linspace(y_min - padding, y_max + padding, gridsize)

    # p(y) and its log once
    kde_y      = gaussian_kde(y_vals)
    p_y_grid   = np.clip(kde_y(grid), eps, None)
    log_p_y    = np.log(p_y_grid)

    # p(x) for every category, with log once
    p_x        = df[col].value_counts(normalize=True)
    log_p_x    = np.log(p_x)

    # ------------------  Loop over each category of X  -----------------------
    mi = 0.0
    for x, log_px in log_p_x.items():
        subset = df.loc[df[col] == x, target_col]
        if len(subset.unique()) < 2:
            continue
        # KDE for p(y | x)  ≡  p(x,y) / p(x)
        kde_xy     = gaussian_kde(df.loc[df[col] == x, target_col])
        p_xy_grid = kde_xy(grid)

        p_joint_grid = p_xy_grid * p_x[x]
        p_joint_grid = np.clip(p_joint_grid, eps, None)

        log_p_joint = np.log(p_joint_grid)

        # integrand:  p_xy * (log p_xy − log p_x − log p_y)
        # do *all* log/exp algebra first, then multiply once
        integrand  = p_joint_grid * (log_p_joint - log_px - log_p_y)

        # trapz is deterministic, vectorised, and perfectly fine in 1‑D
        mi         += np.trapezoid(integrand, grid, dx=grid[1] - grid[0])

    return mi


def MI_discrete_continuous_2(df, col, target_col='duration_seconds',
                gridsize=4096,  # power of two → cheap FFT in KDE
                eps=1e-12):
    """
    Mutual information I(col ; target_col) for a *discrete* X=col and
    a *continuous* Y=target_col, using Gaussian KDE + log‑space integration.
    """
    # ------------------  Pre‑compute pieces that do not depend on x  ----------
    y_vals = df[target_col].to_numpy()
    y_min, y_max = y_vals.min(), y_vals.max()
    padding = 0.05 * (y_max - y_min)

    # Fixed grid avoids adaptive sampling where the KDE is unstable
    grid = np.linspace(y_min - padding, y_max + padding, gridsize)

    # p(y) and its log once
    kde_y      = gaussian_kde(y_vals)
    p_y_grid   = np.clip(kde_y(grid), eps, None)
    log_p_y    = np.log(p_y_grid)

    # p(x) for every category, with log once
    p_x        = df[col].value_counts(normalize=True)
    log_p_x    = np.log(p_x)

    # ------------------  Loop over each category of X  -----------------------
    mi = 0.0
    for x, log_px in log_p_x.items():
        subset = df.loc[df[col] == x, target_col]
        if len(subset.unique()) < 2:
            continue
        # KDE for p(y | x)  ≡  p(x,y) / p(x)
        kde_xy     = gaussian_kde(df.loc[df[col] == x, target_col])
        p_xy_grid  = np.clip(kde_xy(grid), eps, None)
        log_p_xy   = np.log(p_xy_grid)

        # integrand:  p_xy * (log p_xy − log p_x − log p_y)
        # do *all* log/exp algebra first, then multiply once
        integrand  = p_x[x] * p_xy_grid * (log_p_xy - log_p_y)

        # trapz is deterministic, vectorised, and perfectly fine in 1‑D
        mi         += np.trapezoid(integrand, grid)
    return mi

def MI_discrete_continuous_3(df, col, target_col='duration_seconds',
                gridsize=4096,  # power of two → cheap FFT in KDE
                eps=1e-12):
    """
    Mutual information I(col ; target_col) for a *discrete* X=col and
    a *continuous* Y=target_col, using Gaussian KDE + log‑space integration.
    """
    # ------------------  Pre‑compute pieces that do not depend on x  ----------
    y_vals = df[target_col].to_numpy()
    y_min, y_max = y_vals.min(), y_vals.max()
    padding = 0.05 * (y_max - y_min)

    # Fixed grid avoids adaptive sampling where the KDE is unstable
    grid = np.linspace(y_min - padding, y_max + padding, gridsize)

    # p(y) and its log once
    kde_y      = gaussian_kde(y_vals)
    p_y_grid   = np.clip(kde_y(grid), eps, None)
    log_p_y    = np.log(p_y_grid)

    # p(x) for every category, with log once
    p_x        = df[col].value_counts(normalize=True)
    log_p_x    = np.log(p_x)

    # ------------------  Loop over each category of X  -----------------------
    mi = 0.0
    for x, log_px in log_p_x.items():
        subset = df.loc[df[col] == x, target_col]
        if len(subset.unique()) < 2:
            continue
        # KDE for p(y | x)  ≡  p(x,y) / p(x)
        kde_xy     = gaussian_kde(df.loc[df[col] == x, target_col])
        p_xy_grid  = np.clip(kde_xy(grid), eps, None)
        log_p_xy   = np.log(p_xy_grid)

        # integrand:  p_xy * (log p_xy − log p_x − log p_y)
        # do *all* log/exp algebra first, then multiply once
        integrand  = p_xy_grid * (log_p_xy - log_p_y)

        # trapz is deterministic, vectorised, and perfectly fine in 1‑D
        mi         += p_x[x] * np.trapezoid(integrand, grid)
    return mi

## Mutual Information : Continuous - Continuous

In [ ]:
def MI_continuous_continuous(df, col, target_col='duration_seconds',
                             gridsize=4096,  # power of two → cheap FFT in KDE
                             eps=1e-12):
    """
    Mutual information I(col ; target_col) for a *continuous* X=col and
    a *continuous* Y=target_col, using Gaussian KDE + log‑space integration.
    """
    # ------------------  Pre‑compute pieces that do not depend on x  ----------
    x_vals = df[col].to_numpy()
    y_vals = df[target_col].to_numpy()
    x_min, x_max = x_vals.min(), x_vals.max()
    y_min, y_max = y_vals.min(), y_vals.max()
    x_padding = 0 * 0.05 * (x_max - x_min)
    y_padding = 0 * 0.05 * (y_max - y_min)

    x_grid = np.linspace(x_min - x_padding, x_max + x_padding, gridsize)
    y_grid = np.linspace(y_min - y_padding, y_max + y_padding, gridsize)

    dx = x_grid[1] - x_grid[0]
    dy = y_grid[1] - y_grid[0]

    kde_x     = gaussian_kde(x_vals)
    kde_y     = gaussian_kde(y_vals)

    p_x_grid  = np.clip(kde_x(x_grid), eps, None)
    p_y_grid  = np.clip(kde_y(y_grid), eps, None)

    log_p_x   = np.log(p_x_grid)
    log_p_y    = np.log(p_y_grid)

    # Bivariate KDE
    kde_xy = gaussian_kde(np.vstack([x_vals, y_vals]))
    x_mesh, y_mesh = np.meshgrid(x_grid, y_grid, indexing='ij')  # Shape: (gridsize, gridsize)
    xy_samples = np.vstack([x_mesh.ravel(), y_mesh.ravel()])
    p_xy = np.clip(kde_xy(xy_samples), eps, None).reshape(gridsize, gridsize)
    log_p_xy = np.log(p_xy)

    # Compute MI: ∬ p(x, y) * (log p(x, y) - log p(x) - log p(y)) dx dy
    log_p_x_mesh = log_p_x[:, np.newaxis]  # shape (gridsize, 1)
    log_p_y_mesh = log_p_y[np.newaxis, :]  # shape (1, gridsize)

    integrand = p_xy * (log_p_xy - log_p_x_mesh - log_p_y_mesh)
    mi = np.sum(integrand) * dx * dy

    return mi


def MI_continuous_continuous_2(df, col, target_col='duration_seconds',
                             gridsize=4096,  # power of two → cheap FFT in KDE
                             eps=1e-12):
    """
    Mutual information I(col ; target_col) for a *continuous* X=col and
    a *continuous* Y=target_col, using Gaussian KDE + log‑space integration.
    """
    # ------------------  Pre‑compute pieces that do not depend on x  ----------
    x_vals = df[col].to_numpy()
    y_vals = df[target_col].to_numpy()
    x_min, x_max = x_vals.min(), x_vals.max()
    y_min, y_max = y_vals.min(), y_vals.max()
    x_padding = 0 * 0.05 * (x_max - x_min)
    y_padding = 0 * 0.05 * (y_max - y_min)

    x_grid = np.linspace(x_min - x_padding, x_max + x_padding, gridsize)
    y_grid = np.linspace(y_min - y_padding, y_max + y_padding, gridsize)

    dx = x_grid[1] - x_grid[0]
    dy = y_grid[1] - y_grid[0]

    kde_x     = gaussian_kde(x_vals)
    kde_y     = gaussian_kde(y_vals)

    p_x_grid  = np.clip(kde_x(x_grid), eps, None)
    p_y_grid  = np.clip(kde_y(y_grid), eps, None)

    log_p_x   = np.log(p_x_grid)
    log_p_y    = np.log(p_y_grid)

    # Bivariate KDE
    kde_xy = gaussian_kde(np.vstack([x_vals, y_vals]))
    x_mesh, y_mesh = np.meshgrid(x_grid, y_grid, indexing='ij')  # Shape: (gridsize, gridsize)
    xy_samples = np.vstack([x_mesh.ravel(), y_mesh.ravel()])
    p_xy = np.clip(kde_xy(xy_samples), eps, None).reshape(gridsize, gridsize)
    log_p_xy = np.log(p_xy)

    @jit(nopython=True)
    def compute_mi(p_xy, log_p_xy, log_p_x, log_p_y, dx, dy):
        gridsize = p_xy.shape[0]
        mi = 0.0
        for i in range(gridsize):
            for j in range(gridsize):
                integrand = p_xy[i, j] * (log_p_xy[i, j] - log_p_x[i] - log_p_y[j])
                mi += integrand
        mi *= dx * dy
        return mi

    # Compute MI using the JIT-optimized integral
    mi = compute_mi(p_xy, log_p_xy, log_p_x, log_p_y, dx, dy)
    return mi


def MI_continuous_continuous_3(df, col, target_col='duration_seconds',
                             gridsize=4096,  # power of two → cheap FFT in KDE
                             eps=1e-12):
    """
    Mutual information I(col ; target_col) for a *continuous* X=col and
    a *continuous* Y=target_col, using Gaussian KDE + log‑space integration.
    """
    # ------------------  Pre‑compute pieces that do not depend on x  ----------
    x_vals = df[col].to_numpy()
    y_vals = df[target_col].to_numpy()
    x_min, x_max = x_vals.min(), x_vals.max()
    y_min, y_max = y_vals.min(), y_vals.max()
    x_padding = 0 * 0.05 * (x_max - x_min)
    y_padding = 0 * 0.05 * (y_max - y_min)

    x_grid = np.linspace(x_min - x_padding, x_max + x_padding, gridsize)
    y_grid = np.linspace(y_min - y_padding, y_max + y_padding, gridsize)

    dx = x_grid[1] - x_grid[0]
    dy = y_grid[1] - y_grid[0]

    kde_x     = gaussian_kde(x_vals)
    kde_y     = gaussian_kde(y_vals)

    p_x_grid  = np.clip(kde_x(x_grid), eps, None)
    p_y_grid  = np.clip(kde_y(y_grid), eps, None)

    log_p_x   = np.log(p_x_grid)
    log_p_y    = np.log(p_y_grid)

    # Bivariate KDE
    kde_xy = gaussian_kde(np.vstack([x_vals, y_vals]))
    x_mesh, y_mesh = np.meshgrid(x_grid, y_grid, indexing='ij')  # Shape: (gridsize, gridsize)
    xy_samples = np.vstack([x_mesh.ravel(), y_mesh.ravel()])
    p_xy = np.clip(kde_xy(xy_samples), eps, None).reshape(gridsize, gridsize)
    log_p_xy = np.log(p_xy)

    # CUDA kernel
    @cuda.jit
    def compute_mi_kernel(p_xy, log_p_xy, log_p_x, log_p_y, partial_sums):
        tx = cuda.threadIdx.x
        ty = cuda.threadIdx.y
        bx = cuda.blockIdx.x
        by = cuda.blockIdx.y
        block_size_x = cuda.blockDim.x
        block_size_y = cuda.blockDim.y
        i = bx * block_size_x + tx
        j = by * block_size_y + ty

        shared = cuda.shared.array(shape=(256,), dtype=np.float64)
        shared_idx = tx + ty * block_size_x

        if i < p_xy.shape[0] and j < p_xy.shape[1]:
            integrand = p_xy[i, j] * (log_p_xy[i, j] - log_p_x[i] - log_p_y[j])
        else:
            integrand = 0.0

        shared[shared_idx] = integrand
        cuda.syncthreads()

        stride = (block_size_x * block_size_y) // 2
        while stride > 0:
            if shared_idx < stride and shared_idx + stride < block_size_x * block_size_y:
                shared[shared_idx] += shared[shared_idx + stride]
            cuda.syncthreads()
            stride //= 2

        if tx == 0 and ty == 0:
            partial_sums[bx, by] = shared[0]

    # Configure execution
    block_dim = (16, 16)
    grid_dim = ((gridsize + block_dim[0] - 1) // block_dim[0],
                (gridsize + block_dim[1] - 1) // block_dim[1])

    # Transfer data
    d_p_xy = cuda.to_device(p_xy)
    d_log_p_xy = cuda.to_device(log_p_xy)
    d_log_p_x = cuda.to_device(log_p_x)
    d_log_p_y = cuda.to_device(log_p_y)
    d_partial_sums = cuda.device_array(grid_dim, dtype=np.float64)

    # Launch kernel
    compute_mi_kernel[grid_dim, block_dim](d_p_xy, d_log_p_xy, d_log_p_x, d_log_p_y, d_partial_sums)

    # Compute result
    partial_sums_host = d_partial_sums.copy_to_host()
    mi = np.sum(partial_sums_host) * dx * dy
    return mi

to_benchmark = lambda f, n : f(transformed_event_log, 'concept:name', gridsize=n)
l = [MI_discrete_continuous, MI_discrete_continuous_2, MI_discrete_continuous_3]


for i in [2**n for n in range(8, 14)]:
    for f in l:
        start_time = time.perf_counter()
        bf = to_benchmark(f, i)
        end_time = time.perf_counter()
        print(f"Function {f.__name__} with gridsize {i} took {end_time - start_time:.4f} seconds and returned {bf}")

### Test on synthetic data

In [ ]:
def generate_bivariate_gaussian(n_samples, rho):
    """Generate bivariate Gaussian data with specified correlation."""
    mean = [0, 0]
    cov = [[1, rho], [rho, 1]]
    data = multivariate_normal.rvs(mean, cov, size=n_samples)
    return data

def true_mi_gaussian(rho):
    """Compute true MI for bivariate Gaussian variables."""
    return -0.5 * np.log(1 - rho**2)

test_data = generate_bivariate_gaussian(10000, 0.5)
test_df = pd.DataFrame(test_data, columns=['X', 'Y'])

In [ ]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    print(f"n_neighbors={n}: {mutual_info_regression(test_df[['X']], test_df['Y'], n_neighbors=n)}")


In [ ]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    x_discrete = pd.cut(test_df['X'], bins=n, labels=False)
    y_discrete = pd.cut(test_df['Y'], bins=n, labels=False)

    # Compute MI
    print(f"bins={n}: {mutual_info_score(x_discrete, y_discrete)}")

In [ ]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    print(f"bins={n}: {MI_continuous_continuous(test_df, 'X', 'Y', n)}")

In [ ]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    print(f"bins={n}: {MI_continuous_continuous_2(test_df, 'X', 'Y', n)}")

In [ ]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    print(f"bins={n}: {MI_continuous_continuous_3(test_df, 'X', 'Y', n)}")

### Test on real world data

#### Continuous - Continuous

In [ ]:
for n in [2, 4, 8, 16, 32, 64, 128, 256]:
    print(f"bins={n}: {MI_continuous_continuous(transformed_event_log, 'seconds_in_day', 'duration_seconds', n)}")

In [ ]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]:
    print(f"n_neighbors={n}: {mutual_info_regression(transformed_event_log[['seconds_in_day']], transformed_event_log['duration_seconds'], n_neighbors=n)}")

In [ ]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]:
    x_discrete = pd.cut(transformed_event_log['seconds_in_day'], bins=n, labels=False)
    y_discrete = pd.cut(transformed_event_log['duration_seconds'], bins=n, labels=False)

    # Compute MI
    print(f"bins={n}: {mutual_info_score(x_discrete, y_discrete)}")

#### Discrete - Continuous

In [ ]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    print(f"n_neighbors={n}: {MI_discrete_continuous(transformed_event_log, 'concept:name', 'case:RequestedAmount_start', n)}")

In [ ]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    y_discrete = pd.cut(transformed_event_log['case:RequestedAmount_start'], bins=n, labels=False)

    # Compute MI
    print(f"bins={n}: {mutual_info_score(transformed_event_log['concept:name'], y_discrete)}")

In [ ]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    print(f"n_neighbors={n}: {MI_discrete_continuous(transformed_event_log, 'concept:name', 'duration_seconds', n)}")

In [ ]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    y_discrete = pd.cut(transformed_event_log['duration_seconds'], bins=n, labels=False)

    # Compute MI
    print(f"bins={n}: {mutual_info_score(transformed_event_log['concept:name'], y_discrete)}")

#### Discrete - Discrete

In [ ]:
MI_discrete_discrete(transformed_event_log, 'concept:name', 'org:resource_start')

In [ ]:
MI_discrete_discrete(transformed_event_log, 'concept:name', 'case:LoanGoal_start')

In [ ]:
MI_discrete_discrete(transformed_event_log, 'org:resource_start', 'case:LoanGoal_start')

In [ ]:
mutual_info_score(transformed_event_log['concept:name'], transformed_event_log['org:resource_start'])